# correlation analysis of capabilities

In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
# access different dir
import sys
sys.path.insert(1, '../results')

In [ ]:
df_all_devices = pd.read_csv('../results/handle_req_without_gg.csv')

In [ ]:
df_transposed = df_all_devices.set_index('Capability').T

In [ ]:
df_transposed.head()

In [ ]:
replace_map = {
    "T" : 0.5,
    "F" : 0
}
df_transposed_numeric = df_transposed.replace(replace_map).astype(float)
df_transposed_numeric = df_transposed_numeric.fillna(-1)

df_all_devices_numeric = df_all_devices[df_all_devices.columns[1:]].replace(replace_map).astype(float)
df_all_devices_numeric = df_all_devices_numeric.fillna(-1)

In [ ]:
corr = df_transposed_numeric.corr()

In [ ]:
corr = corr.fillna(0)
corr

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.clustermap(corr, cmap="coolwarm", center=0)
plt.show()

In [ ]:
df_transposed_numeric.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(df_transposed_numeric)


In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

In [ ]:
np.cumsum(pca.explained_variance_ratio_)

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=df_transposed_numeric.columns,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

In [ ]:
loadings["PC1"].sort_values(key=abs, ascending=False).head(10)

In [ ]:
feature_groups = loadings.abs().idxmax(axis=1)

In [ ]:
print(feature_groups.to_string())

## feature clustering

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
Z = linkage(X_scaled.T, method="ward")
fig, ax = plt.subplots(figsize=(30, 20))
ax = dendrogram(Z, labels=df_transposed_numeric.columns, orientation='right')
plt.show()